# w14_gpu1_ppt_o2_m0_m3_and_fixed_conformal_v2
Inputs: `shortpose-wave1-private-inputs`


In [ ]:
import hashlib, os, subprocess, sys, tempfile, zipfile
from pathlib import Path
import torch

WORK, MOUNT = Path('/kaggle/working'), Path('/kaggle/input')
assert WORK.is_dir() and torch.cuda.is_available(), 'Kaggle GPU required'
REV = 'b9e60c5ce23551b6612a2f0414b18096758307c9'
with tempfile.TemporaryDirectory(prefix='w14_ppt_v2_') as ts:
    tmp = Path(ts)
    repo, meta_dir = tmp / 'repo', tmp / 'pinns_meta'
    subprocess.run(['git', 'init', '-q', str(repo)], check=True)
    subprocess.run(['git', '-C', str(repo), 'fetch', '-q', '--depth=1', 'https://github.com/pjyyyds/Unstructured-ShortPose-Pred.git', REV], check=True)
    subprocess.run(['git', '-C', str(repo), 'checkout', '-q', '--detach', 'FETCH_HEAD'], check=True)
    env = {**os.environ, 'PYTHONPATH': str(repo), 'PYTHONDONTWRITEBYTECODE': '1'}
    joint_bin = next(MOUNT.rglob('joint_actions_all.bin'))
    heldout_bin = next(MOUNT.rglob('shortpose_heldout.bin'))
    map_bin = next(MOUNT.rglob('pinns_source_label_causal_maps.bin'))
    pose_npz = tmp / 'pose.npz'
    with zipfile.ZipFile(heldout_bin) as zf:
        pose_npz.write_bytes(zf.read('pose.npz'))
    subprocess.run([sys.executable, '-m', 'scripts.stage_pinns_metadata', '--output-dir', str(meta_dir)], cwd=repo, env=env, check=True)
    dest = WORK / 'ppt_mainline_o2_m0_m3_and_fixed_conformal.zip'
    subprocess.run([sys.executable, '-u', '-m', 'scripts.run_ppt_mainline_o2_m0_m3_and_fixed_conformal', str(meta_dir), str(pose_npz), str(joint_bin), str(map_bin), str(dest)], cwd=repo, env=env, check=True)
    print('DONE:', dest, dest.stat().st_size, hashlib.sha256(dest.read_bytes()).hexdigest(), flush=True)
